<a href="https://colab.research.google.com/github/rudalshan0412-code/Intent_Classifier-RAG_Chatbot/blob/main/04)_%EC%B1%97%EB%B4%87_%EB%82%B4_%EA%B2%80%EC%83%89_%EA%B8%B0%EB%8A%A5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# 시작 전 파일 확인

%cd /content/drive/MyDrive/rag_intent_chatbot

!pwd

!find . -maxdepth 3 -type f | sort

/content/drive/MyDrive/rag_intent_chatbot
/content/drive/MyDrive/rag_intent_chatbot
./data/documents/sample.txt
./data/intents.json
./main.py
./models/intent_classifier.pt
./requirements.txt
./src/chatbot.py
./src/__init__.py
./src/intent/dataset.py
./src/intent/__init__.py
./src/intent/model.py
./src/intent/predict.py
./src/intent/train.py
./src/__pycache__/__init__.cpython-312.pyc
./src/rag/chunker.py
./src/rag/document_loader.py
./src/rag/embedder.py
./src/rag/__init__.py
./src/rag/retriever.py
./src/rag/text_preprocessor.py
./src/rag/vector_store.py


In [ ]:
# 전체 파일 목록 확인

from pathlib import Path

required_paths = [
    Path("data/documents/sample.txt"),
    Path("src/rag/document_loader.py"),
    Path("src/rag/text_preprocessor.py"),
    Path("src/rag/chunker.py"),
    Path("src/rag/retriever.py"),
    Path("models/intent_classifier.pt"),
]

for path in required_paths:
    if path.exists():
        size_bytes = path.stat().st_size
        print(f"[존재] {path} - {size_bytes:,} bytes")
    else:
        print(f"[없음] {path}")

[존재] data/documents/sample.txt - 6,398 bytes
[존재] src/rag/document_loader.py - 2,537 bytes
[존재] src/rag/text_preprocessor.py - 1,315 bytes
[존재] src/rag/chunker.py - 6,281 bytes
[존재] src/rag/retriever.py - 0 bytes
[존재] models/intent_classifier.pt - 44,237 bytes


In [ ]:
# 기존 인터페이스 import 확인

from src.rag.document_loader import (
    Document,
    load_documents,
    load_text_file,
)

from src.rag.text_preprocessor import (
    normalize_whitespace,
    preprocess_text,
)

from src.rag.chunker import (
    Chunk,
    chunk_document,
    chunk_documents,
    chunk_text,
)

print("Document:", Document)
print("Chunk:", Chunk)
print("기존 RAG 모듈 import 성공")

Document: <class 'src.rag.document_loader.Document'>
Chunk: <class 'src.rag.chunker.Chunk'>
기존 RAG 모듈 import 성공


In [ ]:
# 필요한 패키지 설치(Colab 기준 Pytorch와 Numpy는 설치되어 있다.)

!pip install -q -U sentence-transformers

# 설치 확인

import numpy as np
import sentence_transformers
import torch

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 22.3 MB/s eta 0:00:00
NumPy: 2.0.2
PyTorch: 2.11.0+cpu
Sentence Transformers: 5.6.1


In [ ]:
!cat requirements.txt

# sentence-transformer 추가(중복 없이)
# sentence-transformer는 문서를 컴퓨터가 이해할 수 있는 숫자배열(벡터)로 변환해준다

from pathlib import Path

requirements_path = Path("requirements.txt") # 경로파일로 변환
requirements_text = requirements_path.read_text(encoding="utf-8") # .read_text는 파일을 읽어서 문자열로 반환함(Path 내 기능)

package_name = "sentence-transformers"

existing_packages = [
    line.strip().split("==")[0].split(">=")[0] # 보통 버전 정보를 == 0.1.0 형식으로 사용하기에 때준다(>= 또한 마찬가지) + 패키지 이름이 인덱스상 위치가 0이다
    for line in requirements_text.splitlines() # 줄바꿈 기준으로 나누어 리스트로 만듦
    if line.strip() and not line.strip().startswith("#") # if 공백을 지웠을때 비었는지 유무 & #(주석)으로 시작하는 경우
]
# sentence=transformers 작성(파일에 작성시 pip 명령어를 통해 전부 설치)
if package_name not in existing_packages: # "sentence-transformers"가 존재하지 않을 때
    with requirements_path.open("a", encoding="utf-8") as file:
        if requirements_text and not requirements_text.endswith("\n"):
            file.write("\n")
        file.write(f"{package_name}\n")

print(requirements_path.read_text(encoding="utf-8"))

sentence-transformers
sentence-transformers



In [ ]:
'''
임베드 모델은 sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2를 사용할 예정이다.

텍스트를 벡터로 변환한다는 뜻은 텍스트를 평가하는 기준을 n개(n차원)만큼 둔다는 뜻이다(이 경우에는 384개)
'''

'\n임베드 모델은 sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2를 사용할 예정이다.\n'

In [ ]:
'''
벡터 정규화 정책은 normalize_embeddings=True 옵션을 사용할 예정이다.
해당 옵션은 임베딩 벡터의 L2 크기를 1로 만든다. 이를 통해 코사인 유사도를 단순한 내적으로 구할 수 있다.
'''

'\n벡터 정규화 정책은 normalize_embeddings=True 옵션을 사용할 예정이다.\n해당 옵션은 임베딩 벡터의 L2 크기를 1로 만든다. 이를 통해 코사인 유사도를 단순한 내적으로 구할 수 있다.\n'

In [ ]:
# embedder.py

%%writefile src/rag/embedder.py
from __future__ import annotations

import numpy as np
from sentence_transformers import SentenceTransformer


DEFAULT_MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)


class TextEmbedder:
    """텍스트를 L2 정규화된 NumPy 임베딩으로 변환한다."""

    def __init__(
        self,
        model_name: str = DEFAULT_MODEL_NAME,
        device: str | None = None,
    ) -> None:
        self.model_name = model_name
        self.model = SentenceTransformer(
            model_name,# 사용할 임베딩 모델의 이름
            device=device, # 어떤 장치에서 처리할지
        )

        self.device = str(self.model.device)

        dimension = self.model.get_sentence_embedding_dimension() # 출력하는 벡터(임베딩)의 차원 수(크기)를 반환

        if dimension is None:
            raise ValueError("임베딩 차원을 확인할 수 없습니다.")

        self.embedding_dimension = dimension

    def encode_text(self, text: str) -> np.ndarray:
        """하나의 문자열을 1차원 임베딩 벡터로 변환한다."""
        if not text.strip():
            raise ValueError("임베딩할 텍스트가 비어 있습니다.")

        embeddings = self.encode_texts([text])

        return embeddings[0]

    def encode_texts(self, texts: list[str]) -> np.ndarray:
        """여러 문자열을 2차원 임베딩 배열로 변환한다."""
        if not texts:
            raise ValueError("임베딩할 텍스트 목록이 비어 있습니다.")

        if any(not text.strip() for text in texts):
            raise ValueError("텍스트 목록에 빈 문자열이 포함되어 있습니다.")

        embeddings = self.model.encode( # 텍스트를 벡터로 변환(이를 (1, 384) 행렬로 반환(이 또한 2차원 배열이다))(이때 문장의 개수만큼 반환하기에 (n, 384)임으로 2차원 배열이다.
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True, # 벡터의 길이를 1로 고정
            show_progress_bar=False,
        )

        return embeddings.astype(np.float32)

Overwriting src/rag/embedder.py


In [ ]:
# embedder.py 개별 테스트

'''
다음 상황을 확인

-모델을 정상적으로 다운로드하는지
-한 문장이 1차원 벡터로 변환되는지
-여러 문장이 2차원 배열로 변환되는지
-벡터의 크기가 1로 정규화되는지
-빈 문자열 오류가 발생하는지
'''
# 모델 다운로드
from src.rag.embedder import TextEmbedder

embedder = TextEmbedder()

print("모델 이름:", embedder.model_name)
print("실행 장치:", embedder.device)
print("임베딩 차원:", embedder.embedding_dimension)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

모델 이름: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
실행 장치: cpu
임베딩 차원: 384


/content/drive/MyDrive/rag_intent_chatbot/src/rag/embedder.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension() # 출력하는 벡터(임베딩)의 차원 수(크기)를 반환


In [ ]:
# 개별 테스트

import numpy as np

single_embedding = embedder.encode_text(
    "임베딩은 텍스트의 의미를 숫자 벡터로 표현합니다."
)

print("자료형:", type(single_embedding))
print("배열 shape:", single_embedding.shape)
print("벡터 크기:", np.linalg.norm(single_embedding))
print("앞쪽 값 일부:", single_embedding[:5])

자료형: <class 'numpy.ndarray'>
배열 shape: (384,)
벡터 크기: 0.99999994
앞쪽 값 일부: [-0.00911417 -0.02076886 -0.00270667  0.013923   -0.02644248]


In [ ]:
# 여러문장 테스트

test_texts = [
    "임베딩은 텍스트를 벡터로 변환합니다.",
    "문서를 작은 Chunk로 분리합니다.",
    "오늘 점심 메뉴는 김치찌개입니다.",
]

test_embeddings = embedder.encode_texts(test_texts)

print("배열 shape:", test_embeddings.shape)
print("각 벡터 크기:", np.linalg.norm(test_embeddings, axis=1))

배열 shape: (3, 384)
각 벡터 크기: [1. 1. 1.]


In [ ]:
# 의미가 가까운 문장이 더 유사한지 테스트

embedding_a = embedder.encode_text(
    "임베딩은 문장의 의미를 숫자 벡터로 표현합니다."
)

embedding_b = embedder.encode_text(
    "문장을 의미를 가진 숫자 배열로 변환하는 것이 임베딩입니다."
)

embedding_c = embedder.encode_text(
    "오늘 서울의 날씨는 맑습니다."
)

similarity_ab = embedding_a @ embedding_b # 기존에 길이를 1로 통일 시켜놓았기에 코사인 유사도 공식의 분모가 1로 고정된다.(따라서 두 벡터의 내적값이 곧 유사도임)
similarity_ac = embedding_a @ embedding_c

print("임베딩 설명 문장끼리:", float(similarity_ab))
print("임베딩 문장과 날씨 문장:", float(similarity_ac))

임베딩 설명 문장끼리: 0.8647962808609009
임베딩 문장과 날씨 문장: -0.014363747090101242


In [ ]:
# 빈 문자열 검증

try:
    embedder.encode_text("   ")
except ValueError as error:
    print("정상적으로 오류 발생:", error)

정상적으로 오류 발생: 임베딩할 텍스트가 비어 있습니다.


In [ ]:
# vector_store.py

%%writefile src/rag/vector_store.py
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from src.rag.chunker import Chunk


@dataclass
class SearchResult:
    """벡터 검색 결과 한 개를 표현한다."""

    chunk: Chunk
    score: float
    rank: int


class VectorStore:
    """Chunk와 정규화된 임베딩을 저장하고 검색한다."""

    def __init__(self) -> None:
        self.chunks: list[Chunk] = []
        self.embeddings: np.ndarray | None = None

    def add(
        self,
        chunks: list[Chunk],
        embeddings: np.ndarray,
    ) -> None:
        """Chunk와 임베딩을 같은 순서로 저장한다."""
        if not chunks:
            raise ValueError("저장할 Chunk가 없습니다.")

        if embeddings.ndim != 2:
            raise ValueError("embeddings는 2차원 배열이어야 합니다.")

        if len(chunks) != embeddings.shape[0]:
            raise ValueError(
                "Chunk 개수와 임베딩 행 개수가 다릅니다."
            )

        embeddings = embeddings.astype(np.float32) # Numpy 배열의 데이터 타입을 32비트 float32로 변환

        if self.embeddings is None:
            self.chunks = list(chunks)# .add가 호출되는 순간 값이 변화한다.
            self.embeddings = embeddings.copy() # 굳이 copy()를 쓸 필요는 없으나 방어적으로 코딩하는경우 사용
            return

        if self.embeddings.shape[1] != embeddings.shape[1]:
            raise ValueError("기존 임베딩과 차원이 다릅니다.")

        self.chunks.extend(chunks)
        self.embeddings = np.vstack( # 배열들을 세로 방향으로 합침
            [self.embeddings, embeddings]
        )

    def search(
        self,
        query_embedding: np.ndarray,
        top_k: int = 3,
    ) -> list[SearchResult]:
        """코사인 유사도가 높은 Top-K Chunk를 반환한다."""
        if self.embeddings is None or not self.chunks:
            raise ValueError("VectorStore가 비어 있습니다.")

        if top_k < 1:
            raise ValueError("top_k는 1 이상이어야 합니다.")

        query_embedding = np.asarray( # Numpy 배열로 변환
            query_embedding,
            dtype=np.float32,
        ).reshape(-1) # 1차원 배열로 나열함

        if query_embedding.shape[0] != self.embeddings.shape[1]: # numpy 배열에서 인데스 0은 행의개수(1번째 차원), 1은 열의 개수(2번째 차원을 의미한다)
            raise ValueError("질문과 Chunk 임베딩 차원이 다릅니다.")

        scores = self.embeddings @ query_embedding # 행렬곱셈
        result_count = min(top_k, len(self.chunks))

        top_indices = np.argsort(scores)[::-1][:result_count]
        # np.argsort()는 오름차순으로 정렬했을때 기존의 인덱스를 반환(이 경우에는 내림차순).

        return [
            SearchResult(
                chunk=self.chunks[index],
                score=float(scores[index]),
                rank=rank,
            )
            for rank, index in enumerate(
                top_indices,
                start=1,
            )
        ]




Overwriting src/rag/vector_store.py


In [ ]:
# 저장 확인

!sed -n '1,260p' src/rag/vector_store.py

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from src.rag.chunker import Chunk


@dataclass
class SearchResult:
    """벡터 검색 결과 한 개를 표현한다."""

    chunk: Chunk
    score: float
    rank: int


class VectorStore:
    """Chunk와 정규화된 임베딩을 저장하고 검색한다."""

    def __init__(self) -> None:
        self.chunks: list[Chunk] = []
        self.embeddings: np.ndarray | None = None

    def add(
        self,
        chunks: list[Chunk],
        embeddings: np.ndarray,
    ) -> None:
        """Chunk와 임베딩을 같은 순서로 저장한다."""
        if not chunks:
            raise ValueError("저장할 Chunk가 없습니다.")

        if embeddings.ndim != 2:
            raise ValueError("embeddings는 2차원 배열이어야 합니다.")

        if len(chunks) != embeddings.shape[0]:
            raise ValueError(
                "Chunk 개수와 임베딩 행 개수가 다릅니다."
            )

        embeddings = embeddings.astype(np.float32) # Numpy 배열의 데이터 타입을 32비트 float32로 변환

        if self.embeddings is None:
            s

In [ ]:
# 개별 테스트

# 가짜 chunk 생성
from src.rag.chunker import Chunk
from src.rag.vector_store import VectorStore

fake_chunks = [
    Chunk(
        text="임베딩은 텍스트를 숫자 벡터로 표현합니다.",
        chunk_id="test_chunk_0001",
        metadata={"file_name": "test.txt"},
        source="test.txt",
        start_index=0,
        end_index=25,
    ),
    Chunk(
        text="Chunk는 긴 문서를 작은 조각으로 나눈 것입니다.",
        chunk_id="test_chunk_0002",
        metadata={"file_name": "test.txt"},
        source="test.txt",
        start_index=26,
        end_index=55,
    ),
    Chunk(
        text="Intent Classifier는 질문 의도를 분류합니다.",
        chunk_id="test_chunk_0003",
        metadata={"file_name": "test.txt"},
        source="test.txt",
        start_index=56,
        end_index=83,
    ),
]

In [ ]:
# 가짜 임베딩 생성

import numpy as np

fake_embeddings = np.array(
    [
        [1.0, 0.0, 0.0],
        [0.0, 1.0, 0.0],
        [0.0, 0.0, 1.0],
    ],
    dtype=np.float32,
)

fake_query_embedding = np.array(
    [0.9, 0.1, 0.0],
    dtype=np.float32,
)

fake_query_embedding /= np.linalg.norm(
    fake_query_embedding
)

In [ ]:
# 저장 후 검색

test_vector_store = VectorStore()

test_vector_store.add(
    chunks=fake_chunks,
    embeddings=fake_embeddings,
)

test_results = test_vector_store.search(
    query_embedding=fake_query_embedding,
    top_k=2,
)

for result in test_results:
    print(
        f"{result.rank}위 | "
        f"score={result.score:.4f} | "
        f"{result.chunk.chunk_id}"
    )
    print(result.chunk.text)
    print()

1위 | score=0.9939 | test_chunk_0001
임베딩은 텍스트를 숫자 벡터로 표현합니다.

2위 | score=0.1104 | test_chunk_0002
Chunk는 긴 문서를 작은 조각으로 나눈 것입니다.



In [ ]:
# top_k가 Chunk보다 큰 경우

large_top_k_results = test_vector_store.search(
    query_embedding=fake_query_embedding,
    top_k=10,
)

print("요청한 top_k:", 10)
print("실제 반환 개수:", len(large_top_k_results))

요청한 top_k: 10
실제 반환 개수: 3


In [ ]:
# 잘못된 입력 검사

try:
    test_vector_store.search(
        query_embedding=fake_query_embedding,
        top_k=0,
    )
except ValueError as error:
    print("정상적으로 오류 발생:", error)

정상적으로 오류 발생: top_k는 1 이상이어야 합니다.


In [ ]:
# retriever.py

%%writefile src/rag/retriever.py
from __future__ import annotations

from src.rag.embedder import TextEmbedder
from src.rag.vector_store import SearchResult, VectorStore


class Retriever:
    """자연어 질문과 관련 있는 Chunk를 검색한다."""

    def __init__(
        self,
        embedder: TextEmbedder, # class TextEmbedder 받을 예정
        vector_store: VectorStore, # class VectorStore 받을 예정
    ) -> None:
        self.embedder = embedder
        self.vector_store = vector_store

    def retrieve(
        self,
        query: str,
        top_k: int = 3,
    ) -> list[SearchResult]:
        """질문을 임베딩하고 Top-K 검색 결과를 반환한다."""
        if not query.strip():
            raise ValueError("검색 질문이 비어 있습니다.")

        query_embedding = self.embedder.encode_text(query)

        return self.vector_store.search(
            query_embedding=query_embedding,
            top_k=top_k,
        )


Overwriting src/rag/retriever.py


In [ ]:

# 저장 확인

!sed -n '1,180p' src/rag/retriever.py

from __future__ import annotations

from src.rag.embedder import TextEmbedder
from src.rag.vector_store import SearchResult, VectorStore


class Retriever:
    """자연어 질문과 관련 있는 Chunk를 검색한다."""

    def __init__(
        self,
        embedder: TextEmbedder,
        vector_store: VectorStore,
    ) -> None:
        self.embedder = embedder
        self.vector_store = vector_store

    def retrieve(
        self,
        query: str,
        top_k: int = 3,
    ) -> list[SearchResult]:
        """질문을 임베딩하고 Top-K 검색 결과를 반환한다."""
        if not query.strip():
            raise ValueError("검색 질문이 비어 있습니다.")

        query_embedding = self.embedder.encode_text(query)

        return self.vector_store.search(
            query_embedding=query_embedding,
            top_k=top_k,
        )


In [ ]:
# 통합 테스트

# 모듈 불러오기
import importlib

import src.rag.embedder
import src.rag.vector_store
import src.rag.retriever

importlib.reload(src.rag.embedder)
importlib.reload(src.rag.vector_store)
importlib.reload(src.rag.retriever)

print("새로 저장한 모듈 reload 완료")

새로 저장한 모듈 reload 완료


In [ ]:
# 전체 파이프라인 실행

from src.rag.document_loader import (
    Document,
    load_documents,
)
from src.rag.text_preprocessor import preprocess_text
from src.rag.chunker import chunk_document
from src.rag.embedder import TextEmbedder
from src.rag.vector_store import VectorStore
from src.rag.retriever import Retriever

# 문서 로딩 후 chunk 로 분할

documents = load_documents("data/documents")

all_chunks = []

for document in documents:
    processed_text = preprocess_text(document.text)

    processed_document = Document(
        text=processed_text,
        metadata=document.metadata,
    )

    chunks = chunk_document(
        document=processed_document,
        chunk_size=500,
        chunk_overlap=100,
    )

    all_chunks.extend(chunks)

print("문서 개수:", len(documents))
print("전체 Chunk 개수:", len(all_chunks))

# 정상 작동하는지 일부 확인

for chunk in all_chunks[:3]:
    print("=" * 60)
    print("chunk_id:", chunk.chunk_id)
    print("source:", chunk.source)
    print("start_index:", chunk.start_index)
    print("end_index:", chunk.end_index)
    print("글자 수:", len(chunk.text))
    print("내용:", chunk.text[:200])

문서 개수: 1
전체 Chunk 개수: 9
chunk_id: sample_chunk_0000
source: sample.txt
start_index: 0
end_index: 382
글자 수: 382
내용: RAG의 정의

RAG는 Retrieval-Augmented Generation의 약자로, 검색 증강 생성이라고 부른다. 일반적인 생성형 언어 모델은 학습 과정에서 익힌 지식과 현재 입력된 문맥을 바탕으로 답변을 만든다. 반면 RAG는 사용자의 질문과 관련된 외부 문서를 먼저 검색하고, 검색된 내용을 언어 모델의 입력 문맥에 함께 제공한 뒤 답변을 생성한다
chunk_id: sample_chunk_0001
source: sample.txt
start_index: 284
end_index: 755
글자 수: 471
내용: 수 있다. 이 프로젝트에서는 문서를 읽고, 정리하고, 작은 단위로 나눈 뒤, 질문과 관련된 부분을 찾아 챗봇 답변에 활용하는 전체 과정을 직접 구현한다.

RAG가 필요한 이유

언어 모델은 그럴듯한 문장을 잘 만들지만, 학습하지 않은 최신 정보나 사용자의 개인 문서에 대해서는 정확히 알 수 없다. 또한 학습 데이터에 비슷한 내용이 있더라도 출처가 불분명하
chunk_id: sample_chunk_0002
source: sample.txt
start_index: 657
end_index: 1155
글자 수: 498
내용: 문제 해결 절차를 검색하도록 만들 수 있다. 문서가 수정되었을 때 모델 전체를 다시 학습하지 않고 검색 대상 문서만 갱신할 수 있다는 점도 중요한 장점이다.

문서 청킹의 의미

긴 문서를 그대로 하나의 데이터로 저장하면 사용자의 짧은 질문과 문서 전체를 비교해야 하므로 관련 부분을 정확히 찾기 어렵다. 또한 언어 모델에 문서 전체를 전달하면 입력 토큰이 


In [ ]:
# Chunk 임베딩 생성

chunk_texts = [
    chunk.text
    for chunk in all_chunks
]

embedder = TextEmbedder()

chunk_embeddings = embedder.encode_texts(
    chunk_texts
)

print("Chunk 개수:", len(all_chunks))
print("임베딩 배열 shape:", chunk_embeddings.shape)
print("임베딩 차원:", embedder.embedding_dimension)

# 정상인 경우
assert len(all_chunks) == chunk_embeddings.shape[0]
assert embedder.embedding_dimension == chunk_embeddings.shape[1]

print("Chunk와 임베딩 크기 일치")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/content/drive/MyDrive/rag_intent_chatbot/src/rag/embedder.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension() # 출력하는 벡터(임베딩)의 차원 수(크기)를 반환


Chunk 개수: 9
임베딩 배열 shape: (9, 384)
임베딩 차원: 384
Chunk와 임베딩 크기 일치


In [ ]:
# VectorStore, Retriever 생성

vector_store = VectorStore()

vector_store.add(
    chunks=all_chunks,
    embeddings=chunk_embeddings,
)

retriever = Retriever(
    embedder=embedder,
    vector_store=vector_store,
)

print("VectorStore Chunk 개수:", len(vector_store.chunks))
print("Retriever 생성 완료")

VectorStore Chunk 개수: 9
Retriever 생성 완료


In [ ]:
# 여러 질문 검색

test_queries = [
    "임베딩은 무엇인가요?",
    "문서를 왜 Chunk로 나누나요?",
    "Intent Classifier는 무슨 역할을 하나요?",
    "벡터 유사도 검색은 어떻게 하나요?",
]

# 출력함수

def print_search_results(
    query: str,
    results,
    preview_length: int = 220,
) -> None:
    print("\n" + "=" * 80)
    print(f"질문: {query}")
    print("=" * 80)

    for result in results:
        chunk = result.chunk

        preview = chunk.text.replace("\n", " ")
        preview = preview[:preview_length]

        if len(chunk.text) > preview_length:
            preview += "..."

        print(f"\n{result.rank}위")
        print(f"score: {result.score:.4f}")
        print(f"chunk_id: {chunk.chunk_id}")
        print(f"source: {chunk.source}")
        print(f"Chunk 글자 수: {len(chunk.text)}")
        print(f"내용: {preview}")

In [ ]:
# 각 질문에 대한 Top 3 검색

for query in test_queries:
    results = retriever.retrieve(
        query=query,
        top_k=3,
    )

    print_search_results(
        query=query,
        results=results,
    )


질문: 임베딩은 무엇인가요?

1위
score: 0.3504
chunk_id: sample_chunk_0008
source: sample.txt
Chunk 글자 수: 184
내용: 프롬프트에 넣어 근거 중심의 응답을 만들 수 있다. 현재 단계에서는 이 전체 구조 중 가장 앞부분인 문서 로딩, 공백 및 줄바꿈 전처리, 문단과 문장을 고려한 청킹을 구현한다. 이후에는 임베딩 생성, 벡터 저장소, Retriever, Intent Classifier와 RAG 라우팅, 최종 챗봇 인터페이스 순서로 확장할 예정이다.

2위
score: 0.3425
chunk_id: sample_chunk_0003
source: sample.txt
Chunk 글자 수: 491
내용: 너무 긴 문단이나 문장은 최종적으로 글자 수 기준으로 나눌 수 있으며, 인접 Chunk 사이에 일부 내용을 겹치게 두면 경계 부근의 문맥이 사라지는 문제를 완화할 수 있다.  임베딩의 의미  컴퓨터는 문장의 의미를 사람처럼 직접 이해하지 못하므로, 텍스트를 수치 벡터로 변환하는 과정이 필요하다. 임베딩은 단어, 문장, 문서 조각의 의미적 특징을 여러 차원의 숫자로 표현한 벡터이다. 의미가 비...

3위
score: 0.3012
chunk_id: sample_chunk_0006
source: sample.txt
Chunk 글자 수: 259
내용: Intent Classifier는 사용자의 문장을 greeting, goodbye, thanks, help, bot_info, document_query 중 하나로 분류한다. 최고 예측 확률이 임계값보다 낮으면 fallback으로 처리하며, document_query로 예측된 경우에만 requires_rag 값을 참으로 설정한다. 이 구조를 사용하면 단순 대화와 문서 기반 질의를 구분하여 불...

질문: 문서를 왜 Chunk로 나누나요?

1위
score: 0.5267
chunk_id: sample_chunk_0002
source: sample.txt
Chunk 

In [ ]:
# 빈 질문 오류 확인

try:
    retriever.retrieve(
        query="   ",
        top_k=3,
    )
except ValueError as error:
    print("정상적으로 오류 발생:", error)

정상적으로 오류 발생: 검색 질문이 비어 있습니다.
